# Section 3: BOM Explosion & Cost Rollup (Q19–Q28)

Bill of materials traversal using `explode_bom()` helper and SQL aggregation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import get_session, run_sql, explode_bom, display_bom_tree, bom_to_df
import pandas as pd
conn, ontology = get_session()

## Q19 — List every ingredient in the formula for FORM-BULK-OC-MINT-026

List every ingredient in the formula for FORM-BULK-OC-MINT-026 — ingredient name, sequence, and quantity in kg per batch.

In [ ]:
# Single-level formula lookup — direct SQL on formula_ingredients
run_sql(conn, """
    SELECT fi.sequence, i.ingredient_code, i.name as ingredient_name, fi.quantity_kg
    FROM formula_ingredients fi
    JOIN ingredients i ON fi.ingredient_id = i.id
    WHERE fi.formula_id = (SELECT id FROM formulas WHERE formula_code = 'FORM-BULK-OC-MINT-026')
    ORDER BY fi.sequence
""")

## Q20 — Full BOM explosion for SKU-ORAL-001

Explode the full bill of materials for SKU-ORAL-001 down to raw materials. I need to see the bulk intermediate stage, any premix sub-intermediates, and the final raw ingredient level — the complete tree.

In [ ]:
# Multi-level BOM explosion using explode_bom helper
bom = explode_bom(conn, sku_code='SKU-ORAL-001')
display_bom_tree(bom)

## Q21 — Total Sodium Fluoride needed for 100 batches of SKU-ORAL-001

If we need to produce 100 batches of SKU-ORAL-001, what is the total kg of Sodium Fluoride (ACT-FLUORIDE-001) required? Roll the quantity up through every level of the BOM.

In [ ]:
bom = explode_bom(conn, sku_code='SKU-ORAL-001')
fluoride = [r for r in bom if r['ingredient_code'] == 'ACT-FLUORIDE-001']
total_per_batch = sum(r['cumulative_quantity_kg'] for r in fluoride)
print(f"Sodium Fluoride per batch: {total_per_batch:.6f} kg")
print(f"For 100 batches: {total_per_batch * 100:.4f} kg")
print(f"\nAppears {len(fluoride)} time(s) in BOM through different paths:")
for r in fluoride:
    print(f"  Path: {' -> '.join(r['bom_path'])}, qty: {r['cumulative_quantity_kg']:.6f} kg")

## Q22 — Total raw material cost for a single batch of SKU-PERSONAL-226

What is the total raw material cost for a single batch of SKU-PERSONAL-226? Walk the BOM tree and multiply quantities by each ingredient's cost per kg.

In [ ]:
bom = explode_bom(conn, sku_code='SKU-PERSONAL-226', resolve_costs=True)
df = bom_to_df(bom)
df['line_cost'] = df['cumulative_quantity_kg'] * df['cheapest_unit_cost'].fillna(0)
print(df[['ingredient_code', 'ingredient_name', 'cumulative_quantity_kg', 'cheapest_unit_cost', 'cheapest_supplier', 'line_cost']].to_string(index=False))
total_cost = df['line_cost'].sum()
print(f"\nTotal raw material cost per batch: ${total_cost:.2f}")

## Q23 — Where-used report for Limonene D-Isomer (ACT-FRAGRANCE-002)

Which finished goods SKUs use Limonene D-Isomer (ACT-FRAGRANCE-002) in their formulation — either directly or through a bulk intermediate or premix? I need the full where-used report.

In [ ]:
# Where-used: find all paths from ingredient up to SKUs
# Check direct use in SKU formulas, use in bulk formulas, and use in premix formulas
run_sql(conn, """
    WITH target AS (
        SELECT id FROM ingredients WHERE ingredient_code = 'ACT-FRAGRANCE-002'
    ),
    -- Direct use in SKU-level formulas
    direct_sku AS (
        SELECT DISTINCT s.sku_code, s.name as sku_name, 'direct in SKU formula' as path
        FROM target t
        JOIN formula_ingredients fi ON fi.ingredient_id = t.id
        JOIN formulas f ON fi.formula_id = f.id AND f.bom_level = 0
        JOIN skus s ON f.product_id = s.id
    ),
    -- Use in bulk intermediate formulas -> SKU
    via_bulk AS (
        SELECT DISTINCT s.sku_code, s.name as sku_name,
               'via ' || bi.bulk_code as path
        FROM target t
        JOIN formula_ingredients fi1 ON fi1.ingredient_id = t.id
        JOIN formulas f1 ON fi1.formula_id = f1.id AND f1.bom_level = 1
        JOIN bulk_intermediates bi ON f1.product_id = bi.id
        JOIN formula_ingredients fi0 ON fi0.ingredient_id = bi.id
        JOIN formulas f0 ON fi0.formula_id = f0.id AND f0.bom_level = 0
        JOIN skus s ON f0.product_id = s.id
    ),
    -- Use in premix -> bulk -> SKU
    via_premix AS (
        SELECT DISTINCT s.sku_code, s.name as sku_name,
               'via ' || bi2.bulk_code || ' -> ' || bi1.bulk_code as path
        FROM target t
        JOIN formula_ingredients fi2 ON fi2.ingredient_id = t.id
        JOIN formulas f2 ON fi2.formula_id = f2.id AND f2.bom_level = 2
        JOIN bulk_intermediates bi2 ON f2.product_id = bi2.id
        JOIN formula_ingredients fi1 ON fi1.ingredient_id = bi2.id
        JOIN formulas f1 ON fi1.formula_id = f1.id AND f1.bom_level = 1
        JOIN bulk_intermediates bi1 ON f1.product_id = bi1.id
        JOIN formula_ingredients fi0 ON fi0.ingredient_id = bi1.id
        JOIN formulas f0 ON fi0.formula_id = f0.id AND f0.bom_level = 0
        JOIN skus s ON f0.product_id = s.id
    )
    SELECT * FROM direct_sku
    UNION ALL
    SELECT * FROM via_bulk
    UNION ALL
    SELECT * FROM via_premix
    ORDER BY sku_code
""")

## Q24 — BOM depth for SKU-HOME-376

How many BOM levels deep is the formula tree for SKU-HOME-376? Is it a simple two-level recipe or does it go through premix sub-intermediates?

In [ ]:
bom = explode_bom(conn, sku_code='SKU-HOME-376')
max_level = max(r['level'] for r in bom) if bom else 0
print(f"BOM depth for SKU-HOME-376: {max_level} levels")
has_premix = any(r['level'] > 2 for r in bom)
print(f"Has premix sub-intermediates: {has_premix}")
display_bom_tree(bom)

## Q25 — Intermediates in the BOM tree for SKU-ORAL-005

For SKU-ORAL-005, list all intermediates that appear in the BOM tree between the finished good and the raw materials. Show the intermediate name, its BOM level, and whether it's a primary bulk or a premix.

In [ ]:
# First get the formula and its intermediate ingredients
run_sql(conn, """
    SELECT DISTINCT bi.bulk_code, bi.name as intermediate_name, bi.bom_level,
           CASE WHEN bi.bom_level = 1 THEN 'primary bulk'
                WHEN bi.bom_level = 2 THEN 'premix'
           END as intermediate_type
    FROM skus s
    JOIN formulas f0 ON f0.product_id = s.id AND f0.bom_level = 0
    JOIN formula_ingredients fi0 ON fi0.formula_id = f0.id
    JOIN bulk_intermediates bi ON fi0.ingredient_id = bi.id
    WHERE s.sku_code = 'SKU-ORAL-005'
    
    UNION
    
    -- Also find premixes used by those bulk intermediates
    SELECT DISTINCT bi2.bulk_code, bi2.name, bi2.bom_level,
           'premix' as intermediate_type
    FROM skus s
    JOIN formulas f0 ON f0.product_id = s.id AND f0.bom_level = 0
    JOIN formula_ingredients fi0 ON fi0.formula_id = f0.id
    JOIN bulk_intermediates bi1 ON fi0.ingredient_id = bi1.id AND bi1.bom_level = 1
    JOIN formulas f1 ON f1.product_id = bi1.id AND f1.bom_level = 1
    JOIN formula_ingredients fi1 ON fi1.formula_id = f1.id
    JOIN bulk_intermediates bi2 ON fi1.ingredient_id = bi2.id AND bi2.bom_level = 2
    WHERE s.sku_code = 'SKU-ORAL-005'
    ORDER BY bom_level, bulk_code
""")

## Q26 — Top-volume raw materials across all formulas

Across all formulas in the system, what is the total quantity in kg for each raw ingredient? Rank by total consumption — I want to see our top-volume raw materials.

In [ ]:
# Aggregate formula_ingredients across all formulas, only raw ingredients (not bulk intermediates)
run_sql(conn, """
    SELECT i.ingredient_code, i.name as ingredient_name,
           COUNT(DISTINCT fi.formula_id) as formula_count,
           SUM(fi.quantity_kg) as total_quantity_kg
    FROM formula_ingredients fi
    JOIN ingredients i ON fi.ingredient_id = i.id
    GROUP BY i.ingredient_code, i.name
    ORDER BY total_quantity_kg DESC
""")

## Q27 — Planned vs actual ingredient variance for batch B-001-000021

For batch B-001-000021, compare what the formula said we should have used versus what we actually consumed. Show the ingredient-level variance in kg — planned versus actual.

In [ ]:
# Compare formula spec vs actual batch consumption
run_sql(conn, """
    SELECT i.ingredient_code, i.name as ingredient_name,
           fi.quantity_kg as planned_kg,
           bi_actual.quantity_kg as actual_kg,
           (bi_actual.quantity_kg - fi.quantity_kg) as variance_kg,
           ROUND(100.0 * (bi_actual.quantity_kg - fi.quantity_kg) / NULLIF(fi.quantity_kg, 0), 2) as variance_pct
    FROM batches b
    JOIN formula_ingredients fi ON fi.formula_id = b.formula_id
    LEFT JOIN batch_ingredients bi_actual ON bi_actual.batch_id = b.id
        AND bi_actual.ingredient_id = fi.ingredient_id
    JOIN ingredients i ON fi.ingredient_id = i.id
    WHERE b.batch_number = 'B-001-000021'
    ORDER BY i.ingredient_code
""")

## Q28 — Shared raw ingredients across 3+ formulas

Which raw ingredients appear in three or more different formulas? These shared components are our biggest volume aggregation opportunities for procurement.

In [ ]:
run_sql(conn, """
    SELECT i.ingredient_code, i.name as ingredient_name,
           COUNT(DISTINCT fi.formula_id) as formula_count,
           SUM(fi.quantity_kg) as total_quantity_kg
    FROM formula_ingredients fi
    JOIN ingredients i ON fi.ingredient_id = i.id
    GROUP BY i.ingredient_code, i.name
    HAVING COUNT(DISTINCT fi.formula_id) >= 3
    ORDER BY formula_count DESC, total_quantity_kg DESC
""")

In [ ]:
conn.close()
print("Session closed.")